# Regresion con Datos Temporales (Notebook 02)

En este notebook vas a recorrer un flujo completo de **regresion** para predecir
la **demanda diaria** (unidades vendidas) de un e-commerce.

A diferencia del notebook 01 (clasificacion), aqui trabajamos con datos que tienen
**orden temporal**, lo que cambia varias reglas del juego:

- Generacion de datos sinteticos con estacionalidad, tendencia y eventos.
- EDA enfocado en patrones temporales.
- **Features de rezago (lag)**: usar el pasado para predecir el futuro.
- **TimeSeriesSplit**: split que respeta el orden del tiempo (no aleatorio).
- Pipeline de preprocesamiento + modelo (**XGBoost**).
- Metricas de regresion: MAE, RMSE, R², MAPE.
- **Feature importance**: que variables le importan mas al modelo.

### ¿Por que esto importa en MLOps?

En produccion, tu modelo siempre predice el **futuro** usando datos del **pasado**.
Si entrenas con datos del futuro mezclados (como hace un `train_test_split` aleatorio),
obtienes metricas optimistas que no reflejan el rendimiento real. **TimeSeriesSplit**
simula este escenario correctamente.

---

## ¿Que es data leakage temporal?

En el notebook 01 vimos que el data leakage ocurre cuando el modelo ve informacion
que no deberia ver. En datos temporales hay un tipo especifico:

**Leakage temporal** = entrenar con datos del futuro para predecir el pasado.

Ejemplo:
- Tienes datos de enero a diciembre 2024.
- `train_test_split(random_state=42)` podria poner **julio** en train y **marzo** en test.
- El modelo estaria usando julio (futuro) para aprender a predecir marzo (pasado).
- En produccion esto nunca pasa: solo tienes datos hasta **hoy**.

### Solucion: TimeSeriesSplit

```
Split aleatorio (MAL para datos temporales):
  [ene] [mar] [jul] [sep] → train
  [feb] [may] [oct] [dic] → test
  (mezcla pasado y futuro)

TimeSeriesSplit (BIEN):
  Fold 1: [ene feb mar] → train    [abr]     → test
  Fold 2: [ene feb mar abr] → train    [may]     → test
  Fold 3: [ene feb mar abr may] → train    [jun]     → test
  (siempre entrena con el pasado, evalua con el futuro)
```

Cada fold usa **mas datos de entrenamiento** que el anterior, y siempre
evalua en el periodo inmediatamente siguiente. Exactamente como pasaria
en produccion.

In [ ]:
%load_ext autoreload
%reload_ext autoreload

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from generate_demand_data import DemandGenerator

In [ ]:
generator = DemandGenerator(seed=42, n_days=730)  # 2 años de datos
df = generator.create_dataset()

df.head(10)

### Nota: los datos tambien se guardan como CSV

El metodo `create_dataset()` genera los datos en memoria **y** los guarda en `demanda_diaria.csv`.

```python
# Alternativa: cargar desde CSV
# df = pd.read_csv("demanda_diaria.csv", parse_dates=["date"])
```

Tambien puedes generarlo desde la terminal:
```bash
uv run python generate_demand_data.py
```

## 1) EDA: entender los patrones temporales

En datos temporales, el EDA debe responder:

- ¿Hay **tendencia** (crece o decrece con el tiempo)?
- ¿Hay **estacionalidad** (patrones que se repiten)?
- ¿Hay **eventos atipicos** (picos o caidas inusuales)?
- ¿Cuantos valores faltantes hay?

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Serie temporal completa
fig = px.line(
    df, x="date", y="units_sold",
    title="Demanda diaria (unidades vendidas)",
    labels={"date": "Fecha", "units_sold": "Unidades"},
)
fig.update_traces(line=dict(width=0.8))
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
# Patron semanal: ¿que dias se vende mas?
day_names = ["Lun", "Mar", "Mie", "Jue", "Vie", "Sab", "Dom"]
weekly = df.groupby("day_of_week")["units_sold"].mean().reset_index()
weekly["day_name"] = day_names

fig = px.bar(
    weekly, x="day_name", y="units_sold",
    title="Demanda promedio por dia de la semana",
    labels={"day_name": "Dia", "units_sold": "Unidades (promedio)"},
)
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
# Patron mensual
monthly = df.groupby("month")["units_sold"].mean().reset_index()

fig = px.bar(
    monthly, x="month", y="units_sold",
    title="Demanda promedio por mes",
    labels={"month": "Mes", "units_sold": "Unidades (promedio)"},
    color_discrete_sequence=["coral"],
)
fig.update_layout(template="plotly_white", xaxis=dict(dtick=1))
fig.show()

## 2) Features de rezago (Lag Features)

En datos temporales, las features mas poderosas suelen ser **valores pasados**
de la misma variable que quieres predecir.

El generador ya creo estas columnas:

| Feature | Que representa |
|---|---|
| `units_sold_lag_1` | Demanda de **ayer** |
| `units_sold_lag_7` | Demanda de **hace 7 dias** (mismo dia de la semana) |
| `units_sold_rolling_7` | Promedio de los **ultimos 7 dias** |
| `units_sold_rolling_30` | Promedio de los **ultimos 30 dias** (tendencia) |

### ¿Es data leakage?

**No**, porque los lags usan `.shift()`, que solo mira hacia atras.
`units_sold_lag_1` del dia 15 contiene el valor del dia 14, que ya paso.
En produccion tendrias exactamente la misma informacion disponible.

Lo que SI seria leakage: usar el promedio de **toda** la columna (incluye el futuro)
o no hacer shift (usar el valor del mismo dia).

In [ ]:
# ¿Que tan correlacionados estan los lags con la demanda?
lag_cols = [
    "units_sold",
    "units_sold_lag_1",
    "units_sold_lag_7",
    "units_sold_rolling_7",
    "units_sold_rolling_30",
]

print("Correlacion con units_sold:")
print(df[lag_cols].corr()["units_sold"].sort_values(ascending=False))

## 3) Split temporal (NO aleatorio)

En datos temporales, **nunca** usamos `train_test_split` con shuffle.
En su lugar:

1. **Split simple**: entrenamos con los primeros N meses, evaluamos con los ultimos.
2. **TimeSeriesSplit**: cross-validation que respeta el orden temporal.

Aqui hacemos el split simple primero (para entrenar y evaluar),
y luego usamos `TimeSeriesSplit` para cross-validation.

In [ ]:
# Definir features y target
feature_cols = [
    "day_of_week",
    "month",
    "is_weekend",
    "is_special_event",
    "temperature",
    "has_campaign",
    "campaign_spend",
    "avg_catalog_price",
    "days_since_start",
    "units_sold_lag_1",
    "units_sold_lag_7",
    "units_sold_rolling_7",
    "units_sold_rolling_30",
]

X = df[feature_cols]
y = df["units_sold"]

# Split temporal: ultimos 90 dias para test
split_index = len(df) - 90

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Train: {len(X_train)} dias ({df['date'].iloc[0].date()} → {df['date'].iloc[split_index-1].date()})")
print(f"Test:  {len(X_test)} dias ({df['date'].iloc[split_index].date()} → {df['date'].iloc[-1].date()})")

In [ ]:
# Visualizar el corte
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["date"].iloc[:split_index], y=y_train,
    mode="lines", name="Train", line=dict(width=0.8),
))
fig.add_trace(go.Scatter(
    x=df["date"].iloc[split_index:], y=y_test,
    mode="lines", name="Test", line=dict(width=0.8, color="orange"),
))
fig.add_vline(
    x=df["date"].iloc[split_index], line_dash="dash", line_color="red",
    annotation_text="Corte", annotation_position="top left",
)
fig.update_layout(
    title="Split temporal: Train vs Test",
    xaxis_title="Fecha", yaxis_title="Unidades vendidas",
    template="plotly_white",
)
fig.show()

## 4) Pipeline: preprocesamiento + XGBoost

Usamos **XGBoost** (Extreme Gradient Boosting), un modelo basado en arboles
de decision que es muy popular en la industria por:

- Buen rendimiento con datos tabulares (como los de un e-commerce).
- Maneja valores faltantes internamente (no siempre necesita imputacion).
- No requiere escalado de features (a diferencia de regresion lineal).
- Tiene regularizacion incorporada para evitar overfitting.

Aun asi, usamos un `Pipeline` con `SimpleImputer` como buena practica,
para que el flujo sea explicito y reproducible.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

pipeline = Pipeline(
    steps=[
        # Imputar NaN con la mediana (para temperature, campaign_spend, etc.)
        ("imputer", SimpleImputer(strategy="median")),
        # Modelo XGBoost
        (
            "model",
            XGBRegressor(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.1,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

pipeline.fit(X_train, y_train)
print("Pipeline entrenado.")

## 5) Metricas de regresion

En clasificacion usamos accuracy, precision, recall. En **regresion** medimos
que tan lejos estan las predicciones de los valores reales:

| Metrica | Formula (intuitiva) | Interpretacion |
|---|---|---|
| **MAE** | Promedio de \|error\| | "En promedio me equivoco por X unidades" |
| **RMSE** | Raiz del promedio del error² | Igual que MAE pero penaliza errores grandes |
| **R²** | 1 - (error modelo / error baseline) | 1.0 = perfecto, 0.0 = igual que predecir la media |
| **MAPE** | Promedio de \|error\| / real × 100 | "Me equivoco por X%" |

En la practica:
- **MAE** es la mas facil de comunicar al negocio ("nos equivocamos por ~20 unidades").
- **RMSE** es util si quieres penalizar dias donde el error fue muy grande.
- **R²** te dice si el modelo es mejor que simplemente predecir el promedio historico.
- **MAPE** es util para comparar entre productos con volumenes muy diferentes.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print(f"MAE:  {mae:.1f} unidades")
print(f"RMSE: {rmse:.1f} unidades")
print(f"R²:   {r2:.4f}")
print(f"MAPE: {mape:.1f}%")

In [ ]:
# Visualizar predicciones vs realidad
test_dates = df["date"].iloc[split_index:].values

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=test_dates, y=y_test.values,
    mode="lines", name="Real", line=dict(width=1.2),
))
fig.add_trace(go.Scatter(
    x=test_dates, y=y_pred,
    mode="lines", name="Prediccion", line=dict(width=1.2, dash="dash"),
))
fig.update_layout(
    title=f"Prediccion vs Real (ultimos 90 dias) — MAE={mae:.1f}, R²={r2:.3f}",
    xaxis_title="Fecha", yaxis_title="Unidades vendidas",
    template="plotly_white",
)
fig.show()

In [ ]:
# Distribucion de errores (residuos)
residuals = y_test - y_pred

fig = px.histogram(
    x=residuals, nbins=30,
    title="Distribucion de residuos",
    labels={"x": "Error (real - prediccion)", "count": "Frecuencia"},
)
fig.add_vline(x=0, line_dash="dash", line_color="red")
fig.update_layout(template="plotly_white")
fig.show()

# Scatter: prediccion vs real
fig2 = px.scatter(
    x=y_test, y=y_pred, opacity=0.5,
    title="Prediccion vs Real",
    labels={"x": "Real", "y": "Prediccion"},
)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
fig2.add_trace(go.Scatter(
    x=[min_val, max_val], y=[min_val, max_val],
    mode="lines", line=dict(dash="dash", color="red"), name="Perfecto",
))
fig2.update_layout(template="plotly_white")
fig2.show()

## 6) Cross-Validation con TimeSeriesSplit

En el notebook 01 usamos `cross_val_score` con K folds aleatorios.
Para datos temporales usamos `TimeSeriesSplit`, que **nunca** usa datos
del futuro para entrenar:

```
Fold 1: [======= TRAIN =======][= TEST =]
Fold 2: [=========== TRAIN ===========][= TEST =]
Fold 3: [=============== TRAIN ===============][= TEST =]
Fold 4: [=================== TRAIN ===================][= TEST =]
Fold 5: [======================= TRAIN =======================][= TEST =]
```

Cada fold tiene **mas datos de entrenamiento** que el anterior,
y el test siempre es el periodo **inmediatamente siguiente**.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

tscv = TimeSeriesSplit(n_splits=5)

# Visualizar los splits
print("Splits de TimeSeriesSplit:")
print("-" * 60)
for i, (train_idx, test_idx) in enumerate(tscv.split(X)):
    train_start = df["date"].iloc[train_idx[0]].date()
    train_end = df["date"].iloc[train_idx[-1]].date()
    test_start = df["date"].iloc[test_idx[0]].date()
    test_end = df["date"].iloc[test_idx[-1]].date()
    print(
        f"  Fold {i+1}: Train [{train_start} → {train_end}] ({len(train_idx)} dias)"
        f"  |  Test [{test_start} → {test_end}] ({len(test_idx)} dias)"
    )

In [ ]:
# Cross-validation con TimeSeriesSplit
cv_scores_mae = cross_val_score(
    pipeline,
    X, y,
    cv=tscv,
    scoring="neg_mean_absolute_error",  # sklearn usa negativo para MAE
)

cv_scores_r2 = cross_val_score(
    pipeline,
    X, y,
    cv=tscv,
    scoring="r2",
)

print(f"MAE por fold: {(-cv_scores_mae).round(2)}")
print(f"MAE promedio: {-cv_scores_mae.mean():.2f} (+/- {cv_scores_mae.std():.2f})")
print()
print(f"R² por fold:  {cv_scores_r2.round(4)}")
print(f"R² promedio:  {cv_scores_r2.mean():.4f} (+/- {cv_scores_r2.std():.4f})")

### Comparacion: TimeSeriesSplit vs split aleatorio

Ejecutamos la misma cross-validation pero con KFold aleatorio para
demostrar que los scores son **mas optimistas** cuando no respetas el tiempo.

In [ ]:
from sklearn.model_selection import KFold

# KFold aleatorio (no respeta el tiempo)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

random_cv_r2 = cross_val_score(pipeline, X, y, cv=kf, scoring="r2")
random_cv_mae = cross_val_score(pipeline, X, y, cv=kf, scoring="neg_mean_absolute_error")

print("=" * 50)
print(f"{'Metrica':<12} {'TimeSeriesSplit':>18} {'KFold aleatorio':>18}")
print("=" * 50)
print(f"{'R²':<12} {cv_scores_r2.mean():>18.4f} {random_cv_r2.mean():>18.4f}")
print(f"{'MAE':<12} {-cv_scores_mae.mean():>18.2f} {-random_cv_mae.mean():>18.2f}")
print("=" * 50)
print()
print("Si el KFold aleatorio da mejores scores, es porque esta")
print("'haciendo trampa' al mezclar futuro y pasado. En produccion")
print("obtendrias el rendimiento del TimeSeriesSplit, no del KFold.")

## 7) Feature Importance: ¿que variables importan?

Los modelos de arboles (XGBoost, Random Forest, ExtraTrees) calculan
que tan util fue cada feature para hacer las predicciones.

### ¿Por que es importante en MLOps?

- **Monitoreo**: si una feature importante deja de estar disponible en produccion
  (por ejemplo, el proveedor de datos de clima falla), sabes que el modelo
  va a degradarse.
- **Feature drift**: si la distribucion de una feature importante cambia
  significativamente, es senal de que hay que reentrenar.
- **Debugging**: si el modelo predice mal un dia, puedes ver que features
  contribuyeron mas a esa prediccion.
- **Simplificacion**: si una feature tiene importancia ~0, puedes eliminarla
  y simplificar el pipeline.

In [ ]:
# Extraer feature importance del modelo dentro del pipeline
model = pipeline.named_steps["model"]
importances = model.feature_importances_

fi_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": importances,
}).sort_values("importance", ascending=True)

fig = px.bar(
    fi_df, x="importance", y="feature", orientation="h",
    title="Feature Importance (XGBoost)",
    labels={"importance": "Importancia", "feature": "Feature"},
)
fig.update_layout(template="plotly_white", height=500)
fig.show()

In [ ]:
# Tabla ordenada de mayor a menor
fi_df.sort_values("importance", ascending=False).reset_index(drop=True)

### Interpretacion esperada

Las features de **lag** (demanda pasada) probablemente dominen el ranking.
Esto tiene sentido: la mejor prediccion de la demanda de maniana es
"parecida a la de hoy". Las features de contexto (clima, campania,
evento) agregan informacion adicional pero no son las principales.

**En un ciclo MLOps** esto te dice:
1. Si el pipeline de datos que calcula los lags falla, el modelo se degrada mucho.
2. Si el API de clima cae, el impacto es menor (puedes imputar con la mediana).
3. La feature `days_since_start` captura la tendencia — si reescribis el modelo,
   el conteo debe seguir siendo consistente.

## 8) Comparacion de modelos

En el notebook 01 solo usamos un modelo (Logistic Regression).
Aqui comparamos **tres modelos de arboles** para ver que funciona mejor
con datos temporales tabulares.

| Modelo | Descripcion |
|---|---|
| **XGBoost** | Gradient boosting: arboles que se corrigen entre si secuencialmente |
| **ExtraTrees** | Arboles aleatorios con splits al azar (muy rapido, menos overfitting) |
| **Random Forest** | Arboles aleatorios con splits optimos (clasico, robusto) |

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor

models = {
    "XGBoost": XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    "RandomForest": RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
}

print(f"{'Modelo':<16} {'MAE':>8} {'RMSE':>8} {'R²':>8}")
print("=" * 44)

for name, model in models.items():
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", model),
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    mae_val = mean_absolute_error(y_test, preds)
    rmse_val = np.sqrt(mean_squared_error(y_test, preds))
    r2_val = r2_score(y_test, preds)

    print(f"{name:<16} {mae_val:>8.1f} {rmse_val:>8.1f} {r2_val:>8.4f}")

## 9) Resumen: diferencias clave con el notebook 01

| Aspecto | Notebook 01 (Clasificacion) | Notebook 02 (Regresion temporal) |
|---|---|---|
| **Target** | Binario (0/1: dar promocion) | Continuo (unidades vendidas) |
| **Split** | `train_test_split` aleatorio | Corte temporal (train=pasado, test=futuro) |
| **Cross-val** | `KFold` (aleatorio) | `TimeSeriesSplit` (respeta el tiempo) |
| **Metricas** | Accuracy, Precision, Recall, ROC-AUC | MAE, RMSE, R², MAPE |
| **Modelo** | Logistic Regression | XGBoost, ExtraTrees, Random Forest |
| **Features extra** | Feature engineering manual | Lag features (valores pasados) |
| **Interpretabilidad** | Coeficientes del modelo | Feature importance |

### Conceptos clave de MLOps que practicamos:

1. **Reproducibilidad**: seeds, pipelines, datos versionados.
2. **Data leakage temporal**: por que el split aleatorio miente en datos temporales.
3. **Feature importance**: saber que features monitorear en produccion.
4. **Comparacion de modelos**: siempre probar mas de uno antes de decidir.
5. **Metricas de negocio**: comunicar MAE en unidades, no solo R².

---

## **Actividad**

Prueba:

1. Cambia la estrategia de imputacion (media vs mediana vs constante)
2. Agrega nuevas lag features (lag_14, lag_30, rolling_14)
3. Usa `GridSearchCV` con `TimeSeriesSplit` para buscar los mejores hiperparametros de XGBoost
4. Compara las metricas del `TimeSeriesSplit` vs un `KFold` con mas splits (10 folds)
5. Entrena un modelo **sin** las features de lag. ¿Cuanto empeora?
6. Investiga: ¿que pasa si usas `train_test_split(shuffle=True)` en vez del corte temporal?

Subir a repositorio individual con la solucion.